# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashmita-21/Flyrank-ML-Internship/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


In [1]:
%pip -q install duckdb huggingface_hub


In [2]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [4]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [9]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d
        FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_impressions ELSE 0
                END
            ) AS imp_prev60,

            SUM(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_clicks ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_last30,

            AVG(
                CASE
                    WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                    THEN f.gsc_avg_position
                END
            ) AS pos_prev60

        FROM {TABLES['fact_daily']} f
        CROSS JOIN bounds b

        WHERE f.report_date > b.end_d - INTERVAL 90 DAY

        GROUP BY 1, 2

        HAVING imp_prev60 >= 200
    )
    SELECT *
    FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

114,435 content items with enough history


,client_hash_id,content_hash_id,imp_last30,imp_prev60,clk_last30,pos_last30,pos_prev60
0,client_e547b89c05043229,content_1557a3abbc832229,198.0,302.0,2.0,11.738551,20.795408
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,554.0,0.0,23.401005,24.031194
2,client_e547b89c05043229,content_48537762b74f5b34,208.0,543.0,0.0,28.054512,20.675021
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,424.0,1.0,24.978010,19.623125
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,1459.0,0.0,34.729708,20.874945


### Week 3: Full Dataset Feature Engineering and Client-Based Evaluation

I used DuckDB to query the full FlyRank warehouse without downloading the entire dataset into memory. I created a 90-day feature window, using the most recent 30 days for the outcome and the preceding 60 days for historical impressions. I selected content items with at least 200 impressions during the previous 60 days.

I joined query-level signals, including visible query count, rare impressions share, anonymized impressions share, and top-query share. I also added historical average search position as an additional feature.

I trained a Random Forest classifier to predict whether a content item's latest 30-day impressions had declined by more than 20% compared with the average monthly impressions from the preceding 60 days. I used GroupShuffleSplit to separate training and testing data by client_hash_id.

The model trained on 92,174 rows from 38 clients and was tested on 12,435 rows from 13 different clients. There were zero overlapping clients between the training and testing sets.

The majority-class baseline accuracy was 74.1%, while the Random Forest achieved 82.4% accuracy. For declining pages, precision was 84.7%, recall was 93.2%, and the F1-score was 88.7%.

These results suggest that the model performed better than the majority-class baseline on the held-out clients. However, further evaluation using multiple client-based splits and feature importance analysis would help determine whether the performance is consistent and which features contribute most to the predictions.


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [10]:
qsignals = con.sql(f"""
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_queries,
        ANY_VALUE(rare_impressions_share) AS rare_share,
        ANY_VALUE(anonymized_impressions_share) AS anon_share,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = (
    qsignals['top_query_impressions'] /
    qsignals['kept_impressions']
)

data = features.merge(
    qsignals,
    on='content_hash_id',
    how='left'
)

print(f'Joined: {len(data):,} rows')
data.head()

Joined: 114,435 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev60,clk_last30,pos_last30,pos_prev60,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_1557a3abbc832229,198.0,302.0,2.0,11.738551,20.795408,5.0,0.132000,0.678000,26.0,95.0,0.273684
1,client_e547b89c05043229,content_e1f6d0c859ba9dc4,195.0,554.0,0.0,23.401005,24.031194,4.0,0.130841,0.562083,121.0,230.0,0.526087
2,client_e547b89c05043229,content_48537762b74f5b34,208.0,543.0,0.0,28.054512,20.675021,1.0,0.169108,0.797603,25.0,25.0,1.000000
3,client_e547b89c05043229,content_27b27b5e13d4e6b7,134.0,424.0,1.0,24.978010,19.623125,5.0,0.186380,0.620072,33.0,108.0,0.305556
4,client_e547b89c05043229,content_8c2c3dab1f1e875f,379.0,1459.0,0.0,34.729708,20.874945,12.0,0.056039,0.817193,37.0,233.0,0.158798


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [11]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score

# Define whether impressions declined by more than 20%
# Compare the latest 30 days with the average monthly impressions
# from the previous 60 days.
data['is_declining'] = (
    data['imp_last30'] < 0.8 * (data['imp_prev60'] / 2)
).astype(int)

# Select features, including historical average search position
feature_cols = [
    'imp_prev60',
    'visible_queries',
    'rare_share',
    'anon_share',
    'top_query_share',
    'pos_prev60'
]

model_data = data.dropna(
    subset=feature_cols + ['is_declining', 'client_hash_id']
).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

# Split by client, not by individual content item
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_tr, y_tr)
predictions = model.predict(X_te)

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print("Training clients:", groups.iloc[train_idx].nunique())
print("Testing clients:", groups.iloc[test_idx].nunique())

overlap = (
    set(groups.iloc[train_idx]) &
    set(groups.iloc[test_idx])
)
print("Overlapping clients:", len(overlap))

print(
    "Baseline accuracy:",
    round(max(y_te.mean(), 1 - y_te.mean()), 3)
)
print(
    "Model accuracy:",
    round(accuracy_score(y_te, predictions), 3)
)

print(classification_report(y_te, predictions, digits=3))

Training rows: 92174
Testing rows: 12435
Training clients: 38
Testing clients: 13
Overlapping clients: 0
Baseline accuracy: 0.741
Model accuracy: 0.824
              precision    recall  f1-score   support

           0      0.726     0.515     0.603      3215
           1      0.847     0.932     0.887      9220

    accuracy                          0.824     12435
   macro avg      0.786     0.724     0.745     12435
weighted avg      0.815     0.824     0.814     12435



Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
